# Silver Layer - Python cleaning on Population+GDP

In [1]:
import pandas as pd
from pathlib import Path
from datetime import datetime

In [2]:
ROOT_DIR     = Path("..").resolve()
BRONZE_DIR   = ROOT_DIR / "data" / "bronze" / "world_bank_data"
SILVER_DIR   = ROOT_DIR / "data" / "silver"
REJECTED_DIR = SILVER_DIR / "rejected"

SILVER_DIR.mkdir(parents=True, exist_ok=True)
REJECTED_DIR.mkdir(parents=True, exist_ok=True)

print(ROOT_DIR)
print(BRONZE_DIR)

C:\GitHub projects\ETL mini project\fifa-worldcup-socioeconomic-analysis
C:\GitHub projects\ETL mini project\fifa-worldcup-socioeconomic-analysis\data\bronze\world_bank_data


In [3]:
POP_FILE = BRONZE_DIR / "API_POP" / "API_SP.POP.TOTL_DS2_en_csv_v2_450890.csv"
GDP_FILE = BRONZE_DIR / "API_GDP" / "API_NY.GDP.MKTP.CD_DS2_en_csv_v2_446954.csv"

print(POP_FILE.name)
print(GDP_FILE.name)

API_SP.POP.TOTL_DS2_en_csv_v2_450890.csv
API_NY.GDP.MKTP.CD_DS2_en_csv_v2_446954.csv


In [4]:
df_pop_raw = pd.read_csv(POP_FILE, skiprows=4, encoding="utf-8")
df_gdp_raw = pd.read_csv(GDP_FILE, skiprows=4, encoding="utf-8")

print("POP shape:", df_pop_raw.shape)
print("GDP shape:", df_gdp_raw.shape)
df_pop_raw.head(3)

POP shape: (266, 71)
GDP shape: (266, 71)


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"Population, total",SP.POP.TOTL,54922.0,55578.0,56320.0,57002.0,57619.0,58190.0,...,108735.0,108908.0,109203.0,108587.0,107700.0,107310.0,107359.0,107995.0,NaN,NaN
1,Africa Eastern and Southern,AFE,"Population, total",SP.POP.TOTL,130075728.0,133534923.0,137171659.0,140945536.0,144904094.0,149033472.0,...,640058741.0,657801085.0,675950189.0,694446100.0,713090928.0,731821393.0,750491370.0,769280888.0,NaN,NaN
2,Afghanistan,AFG,"Population, total",SP.POP.TOTL,9035043.0,9214083.0,9404406.0,9604487.0,9814318.0,10036008.0,...,35688935.0,36743039.0,37856121.0,39068979.0,40000412.0,40578842.0,41454761.0,42647492.0,NaN,NaN


In [5]:
print(df_pop_raw.columns.tolist())

['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', 'Unnamed: 70']


In [6]:
YEAR_COLS = [str(y) for y in range(1998, 2019)]

df_pop = df_pop_raw[["Country Name", "Country Code"] + YEAR_COLS].melt(
    id_vars=["Country Name", "Country Code"],
    var_name="year",
    value_name="population"
)

df_pop["year"] = pd.to_numeric(df_pop["year"])

print(df_pop.shape)
df_pop.head()

(5586, 4)


,Country Name,Country Code,year,population
0,Aruba,ABW,1998,88451.0
1,Africa Eastern and Southern,AFE,1998,385505757.0
2,Afghanistan,AFG,1998,19159996.0
3,Africa Western and Central,AFW,1998,260297834.0
4,Angola,AGO,1998,15159370.0


In [7]:
df_gdp = df_gdp_raw[["Country Name", "Country Code"] + YEAR_COLS].melt(
    id_vars=["Country Name", "Country Code"],
    var_name="year",
    value_name="gdp_per_capita_usd"
)

df_gdp["year"] = pd.to_numeric(df_gdp["year"])

print(df_gdp.shape)
df_gdp.head()

(5586, 4)


,Country Name,Country Code,year,gdp_per_capita_usd
0,Aruba,ABW,1998,1.665363e+09
1,Africa Eastern and Southern,AFE,1998,2.687281e+11
2,Afghanistan,AFG,1998,NaN
3,Africa Western and Central,AFW,1998,2.971265e+11
4,Angola,AGO,1998,6.506222e+09


In [8]:
df = df_pop.merge(
    df_gdp[["Country Name", "year", "gdp_per_capita_usd"]],
    on=["Country Name", "year"],
    how="left"
)

print(df.shape)
df.head()

(5586, 5)


,Country Name,Country Code,year,population,gdp_per_capita_usd
0,Aruba,ABW,1998,88451.0,1.665363e+09
1,Africa Eastern and Southern,AFE,1998,385505757.0,2.687281e+11
2,Afghanistan,AFG,1998,19159996.0,NaN
3,Africa Western and Central,AFW,1998,260297834.0,2.971265e+11
4,Angola,AGO,1998,15159370.0,6.506222e+09


In [9]:
df = df.rename(columns={
    "Country Name": "country_name",
    "Country Code": "country_code"
})

print(df.dtypes)
df.head()

country_name              str
country_code              str
year                    int64
population            float64
gdp_per_capita_usd    float64
dtype: object


,country_name,country_code,year,population,gdp_per_capita_usd
0,Aruba,ABW,1998,88451.0,1.665363e+09
1,Africa Eastern and Southern,AFE,1998,385505757.0,2.687281e+11
2,Afghanistan,AFG,1998,19159996.0,NaN
3,Africa Western and Central,AFW,1998,260297834.0,2.971265e+11
4,Angola,AGO,1998,15159370.0,6.506222e+09


In [10]:
df["country_name"] = df["country_name"].astype("string").str.strip().str.title().replace("", pd.NA)
df["country_code"] = df["country_code"].astype("string").str.strip().str.upper().replace("", pd.NA)
df["population"]   = pd.to_numeric(df["population"], errors="coerce")
df["gdp_per_capita_usd"] = pd.to_numeric(df["gdp_per_capita_usd"], errors="coerce")

print(df.dtypes)
print("\nNull counts:")
print(df.isnull().sum())

country_name           string
country_code           string
year                    int64
population            float64
gdp_per_capita_usd    float64
dtype: object

Null counts:
country_name            0
country_code            0
year                    0
population             21
gdp_per_capita_usd    176
dtype: int64


In [11]:
# Filter out non-country rows (World Bank aggregates)
aggregate_codes = {
    "WLD", "EUU", "LCN", "EAS", "SAS", "SSA", "MEA",
    "NAC", "ECS", "OED", "HIC", "MIC", "LMC", "UMC", "LIC"
}

df = df[~df["country_code"].isin(aggregate_codes)].copy()

print(f"Shape after filter: {df.shape}")
print(f"Distinct countries: {df['country_name'].nunique()}")

Shape after filter: (5271, 5)
Distinct countries: 251


In [12]:
# ── Year filter (1998–2018) ────────────────────────────────────────────
df = df[df["year"].between(1998, 2018)].copy()
print(f"Years present : {sorted(df['year'].unique())}")
print(f"Rows : {df.shape}")

Years present : [np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018)]
Rows : (5271, 5)


In [13]:
# ── Country mapping (World Bank names → WC team names) ────────────────
import json as _json

MAPPING_FILE = ROOT_DIR / "data" / "utils" / "country_mapping.json"
with open(MAPPING_FILE, encoding="utf-8") as f:
    country_mapping = _json.load(f)

mapping_lower = {k.strip().lower(): v for k, v in country_mapping.items()}

df["country_name"] = df["country_name"].apply(
    lambda x: mapping_lower.get(str(x).strip().lower(), x) if pd.notna(x) else x
).astype("string")

print(f"Mapping charged : {len(country_mapping)} entrées")
print("Sample mapped countries:")
print(df[df["country_name"].isin(country_mapping.values())][
    ["country_name", "country_code"]].drop_duplicates().head(10))

Mapping charged : 60 entrées
Sample mapped countries:
              country_name country_code
4                   Angola          AGO
9                Argentina          ARG
13               Australia          AUS
17                 Belgium          BEL
24  Bosnia and Herzegovina          BIH
29                  Brazil          BRA
37             Switzerland          CHE
39                   Chile          CHL
40                   China          CHN
41             Ivory Coast          CIV


In [14]:
# ── UK Nations split ──────────────────────────────────────────────────
UK_SHARES = {
    "England":          0.840,
    "Scotland":         0.084,
    "Wales":            0.047,
    "Northern Ireland": 0.029,
}
UK_CODES = {
    "England":          "ENG",
    "Scotland":         "SCO",
    "Wales":            "WAL",
    "Northern Ireland": "NIR",
}

df_uk = df[df["country_name"] == "United Kingdom"].copy()

if df_uk.empty:
    print("[WARN] 'United Kingdom' introuvable — vérifier le nettoyage")
else:
    nation_dfs = []
    for nation, share in UK_SHARES.items():
        df_n = df_uk.copy()
        df_n["country_name"] = nation
        df_n["country_code"] = UK_CODES[nation]
        df_n["population"]   = (df_uk["population"] * share).round(0)
        nation_dfs.append(df_n)

    df_uk_split = pd.concat(nation_dfs, ignore_index=True)
    df = df[df["country_name"] != "United Kingdom"].copy()
    df = pd.concat([df, df_uk_split], ignore_index=True).reset_index(drop=True)

    print(f"UK → {len(UK_SHARES)} nations × {df_uk['year'].nunique()} years = {len(df_uk_split)} rows")
    print(df_uk_split[["country_name", "country_code", "year", "population", "gdp_per_capita_usd"]].head(8))

UK → 4 nations × 21 years = 84 rows
  country_name country_code  year  population  gdp_per_capita_usd
0      England          ENG  1998  49129198.0        1.660821e+12
1      England          ENG  1999  49293271.0        1.693459e+12
2      England          ENG  2000  49469712.0        1.671598e+12
3      England          ENG  2001  49660525.0        1.656171e+12
4      England          ENG  2002  49871202.0        1.790537e+12
5      England          ENG  2003  50103965.0        2.061228e+12
6      England          ENG  2004  50389840.0        2.429775e+12
7      England          ENG  2005  50737013.0        2.551362e+12


In [15]:
df["review_reason"] = pd.NA

# Priority 1
df.loc[df["country_name"].isna(), "review_reason"] = "MISSING_COUNTRY_NAME"

# Priority 2+
df.loc[df["review_reason"].isna() & df["country_code"].isna(),
       "review_reason"] = "MISSING_COUNTRY_CODE"

df.loc[df["review_reason"].isna() & df["population"].isna(),
       "review_reason"] = "MISSING_POPULATION"

df.loc[df["review_reason"].isna() & df["gdp_per_capita_usd"].isna(),
       "review_reason"] = "MISSING_GDP"

print(df["review_reason"].value_counts(dropna=False))

review_reason
<NA>                  5158
MISSING_GDP            155
MISSING_POPULATION      21
Name: count, dtype: int64


In [16]:
# Check of UK nations splitted

nations = ["England", "Wales", "Scotland", "Northern Ireland"]
print(df[df["country_name"].isin(nations)][["country_name", "country_code", "year"]].drop_duplicates().sort_values("country_name"))

     country_name country_code  year
5250      England          ENG  1998
5269      England          ENG  2017
5268      England          ENG  2016
5267      England          ENG  2015
5266      England          ENG  2014
...           ...          ...   ...
5307        Wales          WAL  2013
5308        Wales          WAL  2014
5309        Wales          WAL  2015
5311        Wales          WAL  2017
5312        Wales          WAL  2018

[84 rows x 3 columns]


In [17]:
# ── Valid / Rejected split ────────────────────────────────────────────
valid    = df[df["review_reason"].isna()].copy()
rejected = df[df["review_reason"].notna()].copy()

print(f"Valid rows    : {len(valid)}")
print(f"Rejected rows : {len(rejected)}")
print("\nRejection breakdown:")
print(rejected.groupby("review_reason").size().reset_index(name="count"))

Valid rows    : 5158
Rejected rows : 176

Rejection breakdown:
        review_reason  count
0         MISSING_GDP    155
1  MISSING_POPULATION     21


In [18]:
from datetime import timezone


# ── Add metadata ──────────────────────────────────────────────────────
valid["source_name"]    = "API_POP + API_GDP"
valid["load_timestamp"] = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

valid.head()

,country_name,country_code,year,population,gdp_per_capita_usd,review_reason,source_name,load_timestamp
0,Aruba,ABW,1998,88451.0,1.665363e+09,<NA>,API_POP + API_GDP,2026-06-30 09:24:39
1,Africa Eastern And Southern,AFE,1998,385505757.0,2.687281e+11,<NA>,API_POP + API_GDP,2026-06-30 09:24:39
3,Africa Western And Central,AFW,1998,260297834.0,2.971265e+11,<NA>,API_POP + API_GDP,2026-06-30 09:24:39
4,Angola,AGO,1998,15159370.0,6.506222e+09,<NA>,API_POP + API_GDP,2026-06-30 09:24:39
5,Albania,ALB,1998,3128530.0,2.600357e+09,<NA>,API_POP + API_GDP,2026-06-30 09:24:39


In [19]:
# ── Save outputs ──────────────────────────────────────────────────────
valid.drop(columns=["review_reason"]).to_csv(
    SILVER_DIR / "valid_socioeconomics.csv", index=False, encoding="utf-8")

rejected.to_csv(
    REJECTED_DIR / "rejected_socioeconomics.csv", index=False, encoding="utf-8")

print("Saved: valid_socioeconomics.csv")
print("Saved: rejected_socioeconomics.csv")

Saved: valid_socioeconomics.csv
Saved: rejected_socioeconomics.csv


In [20]:
# ── Validation summary ────────────────────────────────────────────────
summary_lines = [
    "=== Silver Validation Summary — Socioeconomics ===",
    f"Run timestamp : {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')}",
    "",
    f"POP bronze rows loaded       : {len(df_pop_raw)}",
    f"GDP bronze rows loaded       : {len(df_gdp_raw)}",
    f"Rows after unpivot + merge   : {len(df)}",
    f"Valid rows written           : {len(valid)}",
    f"Rejected rows                : {len(rejected)}",
    "",
    "--- Rejection breakdown ---",
]
for _, row in rejected.groupby("review_reason").size().reset_index(name="count").iterrows():
    summary_lines.append(f"  {row['review_reason']:<30} : {row['count']}")

summary_text = "\n".join(summary_lines)
print(summary_text)

(SILVER_DIR / "validation_summary.txt").write_text(summary_text, encoding="utf-8")
print("\nSaved: validation_summary.txt")

=== Silver Validation Summary — Socioeconomics ===
Run timestamp : 2026-06-30 09:24:40

POP bronze rows loaded       : 266
GDP bronze rows loaded       : 266
Rows after unpivot + merge   : 5334
Valid rows written           : 5158
Rejected rows                : 176

--- Rejection breakdown ---
  MISSING_GDP                    : 155
  MISSING_POPULATION             : 21

Saved: validation_summary.txt
